# Analyse des données du baccalauréat en France

Ce notebook analyse les données du baccalauréat en France à partir des résultats par académie.  
Grâce à un EDA (fait sur un notebook dédié), un preprocessing et modélisation de Machine Learning  
pour prédire les proportions d'élèves avec une mention.

## Import + configuation

In [161]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,  StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from sklearn.linear_model import  LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## Chargement des données déjà nettoyées

In [162]:
path= '../data/clean_dataset.csv'
df = pd.read_csv(path, delimiter=',')

## Construction du dataset de modelisation

Création d'un nouveau dataset (copie du 1er) avec uniquement les colonnes utilisées  
dans notre modèle.

In [163]:
df_ml = df.copy()

column_droped = [ 
    'admis_au_1er_groupe',
    'refuses_au_1er_groupe', 
    'ajournes_passant_les_epreuves_du_rattrapage', 
    'refuses_au_rattrapage',
    'refuses_totaux',
    'admis_au_rattrapage'
]

df_ml = df_ml.drop(columns=column_droped)

Création de colonnes :  
colonne cible avec toutes les mentions  
colonne taux de présence  
colonne proportion des mentions  

In [164]:
column_mention = [
    'mention_tb_avec_les_felicitations',
    'mention_tb_sans_les_felicitations',
    'mention_b',
    'mention_ab', 
    'sans_mention'
]

df_melt = df_ml.melt(
    id_vars=(
        [
            'session',
            'academie', 
            'sexe', 
            'statut_du_candidat',
            'serie',
            'presents',
            'inscrits',
            'admis_totaux']),
    value_vars=(column_mention),
    var_name='mention',
    value_name='nombre'
)

df_melt['taux_presence'] = df_melt['presents'] / df_melt['inscrits']
df_melt['proportion_mention'] = df_melt['nombre'] / df_melt['admis_totaux']
df_melt= df_melt.dropna(subset=['proportion_mention'])

Mapping de la colonne mention pour une simplier les noms

In [165]:
mapping_mention = {
    'mention_ab': 'AB',
    'sans_mention' : 'sans_mention',
    'mention_b' : 'B',
    'mention_tb_sans_les_felicitations' : 'TB',
    'mention_tb_avec_les_felicitations' : 'TB'
}

df_melt['mention'] = df_melt['mention'].map(mapping_mention)

Distribution de la colonne mention

In [166]:
df_target = df_melt['mention'].value_counts().reset_index().sort_values(by='count')
px.bar(df_target, x='mention', y='count')

## Preprocessing

Séparation de la cible Y des features X

In [167]:
target_name = 'proportion_mention'
features = ['session', 'academie', 'sexe', 'statut_du_candidat','serie','taux_presence']

print('Separating labels from features...')
Y = df_melt.loc[:, target_name]
X = df_melt.loc[:, features]
print('...Done.')
print(f'{Y.head()}\n')
print(f'{X.head()}\n')

print('Dividing into train and test sets...')
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42) 
print('...Done.\n')

Separating labels from features...
...Done.
0   0.0000
1   0.0000
2   0.0000
3   0.0179
4   0.0000
Name: proportion_mention, dtype: float64

   session          academie sexe       statut_du_candidat    serie  \
0     2021     aix-marseille    F  enseignement_a_distance  general   
1     2021            amiens    F  enseignement_a_distance  general   
2     2021          besancon    F  enseignement_a_distance  general   
3     2021          bordeaux    F  enseignement_a_distance  general   
4     2021  clermont-ferrand    F  enseignement_a_distance  general   

   taux_presence  
0         0.9459  
1         1.0000  
2         0.9615  
3         0.9683  
4         1.0000  

Dividing into train and test sets...
...Done.



Détection des colonnes numériques et catégorielles

In [168]:
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('\nFound categorical features ', categorical_features)

Found numeric features  ['session', 'taux_presence']

Found categorical features  ['academie', 'sexe', 'statut_du_candidat', 'serie']


Transformation des colonnes catégorielles et numériques  
Application sur train et test set

In [169]:
numeric_transformer = StandardScaler() 
categorical_transformer = OneHotEncoder(drop='first')  

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print('Performing preprocessings on train set...')
print(X_train.head(3))
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:3])

print('\nPerforming preprocessings on test set...')
print(X_test.head(3))
X_test = preprocessor.transform(X_test)  
print('...Done.')
print(X_test[0:3, :])

Performing preprocessings on train set...
       session academie sexe       statut_du_candidat serie  taux_presence
6002      2024    dijon    M                 scolaire  stmg         1.0000
2259      2022  limoges    F               individuel  stav         1.0000
22208     2025  limoges    F  enseignement_a_distance  st2s         1.0000
...Done.
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16 stored elements and shape (3, 51)>
  Coords	Values
  (0, 0)	0.6810326827673778
  (0, 1)	0.5037937595875047
  (0, 8)	1.0
  (0, 31)	1.0
  (0, 35)	1.0
  (0, 50)	1.0
  (1, 0)	-0.7394997175358207
  (1, 1)	0.5037937595875047
  (1, 14)	1.0
  (1, 34)	1.0
  (1, 45)	1.0
  (2, 0)	1.391298882918977
  (2, 1)	0.5037937595875047
  (2, 14)	1.0
  (2, 32)	1.0
  (2, 44)	1.0

Performing preprocessings on test set...
       session   academie sexe statut_du_candidat              serie  \
35281     2024  normandie    F      apprentissage  pro_services_agri   
34021     2023      corse    M      appr

## Modélisation  

Utilisation de 2 modèles :  
- Linear Regression
- Voting Regressor avec 3 modèles :  
Random Forest Regressor  
XGB Regressor 
Gadient Boosting Regressor

### Linear Regression

In [170]:
# Train model
lr = LinearRegression()

print('Training model...')
lr.fit(X_train, Y_train)
print('...Done.')

# Prédictions
print('Predictions on training set...')
Y_train_pred = lr.predict(X_train)
print('...Done.')
print(Y_train_pred[0:5])

print('Predictions on test set...')
Y_test_pred = lr.predict(X_test)
print('...Done.')
print(Y_test_pred[0:5])

# Scores
lr_r2_train = r2_score(Y_train, Y_train_pred)
lr_r2_test = r2_score(Y_train, Y_train_pred)
lr_mae_train = mean_absolute_error(Y_train, Y_train_pred)
lr_mae_test = mean_absolute_error(Y_test, Y_test_pred)
lr_rmse_train = root_mean_squared_error(Y_train, Y_train_pred)
lr_rmse_test = root_mean_squared_error(Y_test, Y_test_pred)

#Print scores
print(f'r2_score on training set : {lr_r2_train}') 
print(f'r2_score on test set : {lr_r2_test}')

print(f'\nmean_absolute_error on training set : {lr_mae_train}')
print(f'mean_absolute_error on test set : {lr_mae_test}')

print(f'\nroot_mean_squared_error on training set : {lr_rmse_train}')
print(f'root_mean_squared_error on test set : {lr_rmse_test}')

Training model...
...Done.
Predictions on training set...
...Done.
[0.20827664 0.19520334 0.20118411 0.2018247  0.20670086]
Predictions on test set...
...Done.
[0.19407515 0.2033689  0.19935606 0.19326626 0.20296366]
r2_score on training set : 0.00042029590648218207
r2_score on test set : 0.00042029590648218207

mean_absolute_error on training set : 0.19529026032055447
mean_absolute_error on test set : 0.1964495597956842

root_mean_squared_error on training set : 0.24772014934380737
root_mean_squared_error on test set : 0.2476287938069418


### Voting Regressor

In [171]:
# Train model
rr = RandomForestRegressor(n_jobs=-1, random_state=42)
xgb = XGBRegressor(random_state=42)
gbr = GradientBoostingRegressor(random_state=42)

model = VotingRegressor(
    estimators=[
        ('rr', rr),
        ('lr', lr),
        ('gbr', gbr)
    ],
    n_jobs=-1
)

print('Training model...')
model.fit(X_train, Y_train)
print('...Done.')

# Prédictions
print('Predictions on training set...')
Y_train_pred = model.predict(X_train)
print('...Done.')
print(Y_train_pred[0:5])

print('Predictions on test set...')
Y_test_pred = model.predict(X_test)
print('...Done.')
print(Y_test_pred[0:5])

# Scores
vr_r2_train = r2_score(Y_train, Y_train_pred)
vr_r2_test = r2_score(Y_train, Y_train_pred)
vr_mae_train = mean_absolute_error(Y_train, Y_train_pred)
vr_mae_test = mean_absolute_error(Y_test, Y_test_pred)
vr_rmse_train = root_mean_squared_error(Y_train, Y_train_pred)
vr_rmse_test = root_mean_squared_error(Y_test, Y_test_pred)

# Print Scores
print(f'r2_score on training set : {vr_r2_train}') 
print(f'r2_score on test set : {vr_r2_test}')

print(f'\nmean_absolute_error on training set : {vr_mae_train}')
print(f'mean_absolute_error on test set : {vr_mae_test}')

print(f'\nroot_mean_squared_error on training set : {vr_rmse_train}')
print(f'root_mean_squared_error on test set : {vr_rmse_test}')

Training model...
...Done.
Predictions on training set...
...Done.
[0.21342636 0.24738445 0.19508035 0.20528723 0.22064142]
Predictions on test set...
...Done.
[0.18395657 0.18491099 0.19303642 0.19514827 0.18509669]
r2_score on training set : 0.03352969835671804
r2_score on test set : 0.03352969835671804

mean_absolute_error on training set : 0.19167838834454992
mean_absolute_error on test set : 0.21251846679548966

root_mean_squared_error on training set : 0.24358294396003155
root_mean_squared_error on test set : 0.2686423734618777


## Comparaison des résultats

In [172]:
model_name = [
    'r2 on train',
    'r2 on test',
    'mae on train',
    'mae on test',
    'rmse on train',
    'rmse on test'
]

linear_regression = [
    lr_r2_train,
    lr_r2_test,
    lr_mae_train,
    lr_mae_test,
    lr_rmse_train,
    lr_rmse_test
]

voting_regressor = [
    vr_r2_train,
    vr_r2_test,
    vr_mae_train,
    vr_mae_test,
    vr_rmse_train,
    vr_rmse_test
]

df_score = pd.DataFrame(
    {
        'model_name' : model_name, 
        'linear_regression' : linear_regression, 
        'voting_regressor' : voting_regressor
    }
)

df_score = df_score.set_index('model_name')
display(df_score)

,linear_regression,voting_regressor
model_name,,
r2 on train,0.0004,0.0335
r2 on test,0.0004,0.0335
mae on train,0.1953,0.1917
mae on test,0.1964,0.2125
rmse on train,0.2477,0.2436
rmse on test,0.2476,0.2686


Le "meilleur" modèle est linear regression mais les performances sont très très basses.

## Interprétation : Feature importance 

In [173]:
column_names = preprocessor.get_feature_names_out()

Nettoyage du nom des colonnes pour une meilleur lisibilité.

In [174]:
column_names = pd.Series(column_names).str.replace('num__','').str.replace('cat__','')

In [175]:
# Create a pandas DataFrame
feature_importance = pd.DataFrame(
    index=column_names,
    data=np.abs(lr.coef_),
    columns=["feature_importances"],
)
feature_importance = feature_importance.sort_values(by="feature_importances")

Mise en forme graphique des coefficients

In [176]:
fig_feature = px.bar(feature_importance, orientation="h")
fig_feature.update_layout(showlegend=False, margin={"l": 120})
fig_feature.update_layout(autosize=False, width=1200, height=600)
fig_feature.show()

Top 5 des coefficients

In [177]:
feature_importance.sort_values(by="feature_importances", ascending=False).head(5)

,feature_importances
academie_bordeaux,0.0085
academie_poitiers,0.0079
academie_montpellier,0.0076
academie_versailles,0.0070
academie_strasbourg,0.0068


## Conclusion  

Il n'y a pas vraiement de variable clés, les features sont proches les unes des autres.  
Biais potentiels : données agrégées, pas de variables individuelles.  
Les modèles montrent des performances quasi inexistantes, modèles peu prédictif.  
Limites : absence de données individuelles, agrégation par groupes.  